In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — TEST DU MODE « FLEX » DE GEMINI SUR DES MATCHS COMPLETS EN REPLAY (12/11/2026)
# UN SEUL CHANGEMENT : SCOUTIA_GEMINI_TIER = "flex" (même modèle 3.8 Flash, mêmes questions, mêmes réglages ; tarif −50 %,
# délai variable). Les bornes ne sont pas rejouées (cache de suivi) ; en production elles restent au tarif standard.
# Mesure : DURÉE de l'analyse Gemini en Flex, nombre de refus / replis au tarif standard, coût, et résultats (buts,
# équipes) comparés aux références en standard (même modèle : les écarts éventuels = variabilité normale de Gemini).
# CRITÈRES FIXÉS AVANT : 1. buts et équipes dans la variabilité habituelle (aucun vrai but perdu par rapport aux
# références ; 0 attribution fausse) ; 2. durée de l'analyse compatible avec une livraison sous 48 h (< 24 h par match) ;
# 3. coût Gemini mesuré ≈ la moitié du standard (facturation à confirmer dans la console Google le lendemain).
# Coût prévu ≈ 1 $ par match. Durée : inconnue (c'est ce qu'on mesure) — commencer par 2 matchs.
# Nécessite gemini_validator.py (SCOUTIA_GEMINI_TIER) et ai/attribution_equipes.py du 12/11 sur GitHub.
# Entrées : codeprojectai (caches), les vidéos, mziparameter, test_resolution_gemini_cellule.py. Pas de GPU.
# ════════════════════════════════════════════════════════════════════════════════════════════
TIER = "flex"
MATCHS_A_TESTER = ["minerois_ster", "spa"]     # 2 matchs pour commencer (référence standard connue : 5/5 et 7/7)
RESOLUTION = "medium"            # validation des coups d'envoi
RESOLUTION_CLASSIF = "low"       # classification des temps forts
REFERENCE = {   # (vrais buts, faux buts) : résolution NORMALE quand connue (spa, minerois_ster, franchimont, raeren),
    # sinon run « medium » seul du 08/11 (andrimont : cache arrêté à 61:39, 5 buts évaluables)
    "spa": (7, 0), "minerois_ster": (5, 1), "franchimont": (5, 4), "raeren": (8, 4), "andrimont": (5, 1),
    "p1_minerois": (3, 1), "goe": (2, 5), "stembert": (3, 3), "wanze": (2, 3), "juprelle": (1, 1)}

import glob, io, re, sys, time, contextlib

_src = next(iter(sorted(glob.glob("/kaggle/input/**/test_resolution_gemini_cellule.py", recursive=True))
                 + glob.glob("/kaggle/working/test_resolution_gemini_cellule.py")), None)
if not _src:
    raise SystemExit("❌ test_resolution_gemini_cellule.py introuvable (l'ajouter aux entrées ou dans /kaggle/working)")
CODE = open(_src, encoding="utf-8").read()
# on retire l'étape tenues + attribution (de « DÉTECTION DES TENUES » jusqu'au calcul du coût réel)
_a = CODE.index('print("\\n" + "=" * 60 + f"\\nDÉTECTION DES TENUES')
_b = CODE.index("# ── coût réel de ce run")
_VERIF_ATTR = '''
# ── 09/11 : vérification de l'ATTRIBUTION FAITE PAR LE PIPELINE (plus par le notebook) ──
import re as _re_v, collections as _col_v
_att_v = (result or {}).get("attribution_equipes") or {}
_ten_v = _att_v.get("tenues") or {}
def _parse_v(t):
    m = _re_v.match(r"maillot (.*) / short (.*)", str(t or ""))
    return (m.group(1), m.group(2)) if m else ("", "")
_TD_v = (_parse_v(_ten_v.get("A")), _parse_v(_ten_v.get("B")))
_mots_v = lambda x: set(str(x).lower().replace(",", " ").replace("/", " ").split()) - {"and", "with", "dark", "light"}
_sc_v = lambda det, vrai: len(_mots_v(det[0]) & _mots_v(vrai[0])) * 2 + len(_mots_v(det[1]) & _mots_v(vrai[1]))
_VT_v = VRAIES_TENUES_TOUTES[MATCH]; _VE_v = VERITE_EQUIPES[MATCH]
_d_v = _sc_v(_TD_v[0], _VT_v[0]) + _sc_v(_TD_v[1], _VT_v[1]); _i_v = _sc_v(_TD_v[0], _VT_v[1]) + _sc_v(_TD_v[1], _VT_v[0])
_map_v = None if _d_v == _i_v else ({"A": "A", "B": "B"} if _d_v > _i_v else {"A": "B", "B": "A"})
_bil_v = _col_v.Counter(); _vu_v = set()
for _b_v in (result or {}).get("buts", []):
    _g_v = float(_b_v.get("time_abs") or 0)
    _m_v = [h for h in _VE_v if abs(int(h.split(":")[0]) * 60 + int(h.split(":")[1]) - _g_v) <= 20 and h not in _vu_v]
    if not _m_v:
        continue
    _vu_v.add(_m_v[0])
    _e_v = _b_v.get("equipe")
    _ok_v = "—" if not _e_v else ("?" if not _map_v else ("juste" if _map_v[_e_v] == _VE_v[_m_v[0]] else "FAUX"))
    _bil_v[(_b_v.get("niveau_equipe") or "non attribué", _ok_v)] += 1
print(f"ATTRIBUTION PIPELINE : tenues {'OK' if _map_v else 'CORRESPONDANCE IMPOSSIBLE'} | "
      f"justes {_bil_v[('confirmé', 'juste')] + _bil_v[('probable', 'juste')]} | "
      f"FAUX {sum(v for k, v in _bil_v.items() if k[1] == 'FAUX')} | "
      f"non attribués {sum(v for k, v in _bil_v.items() if k[0] == 'non attribué')} | "
      f"coût {_att_v.get('cout_dollars', 0)} $ | escalades {_att_v.get('escalades', 0)} | "
      f"recompressées {_att_v.get('requetes_recompressees', 0)} | ÉCHECS {_att_v.get('questions_en_echec', 0)}")
'''
CODE = CODE[:_a] + "\n" + _VERIF_ATTR + "\n" + CODE[_b:]
CODE = re.sub(r'^RESOLUTION = .*$', f'RESOLUTION = "{RESOLUTION}"', CODE, count=1, flags=re.M)
CODE = CODE.replace('os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION',
                    'os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION\n'
                    f'os.environ["SCOUTIA_MEDIA_RESOLUTION_KO"] = "{RESOLUTION}"\n'
                    f'os.environ["SCOUTIA_MEDIA_RESOLUTION_CLASSIF"] = "{RESOLUTION_CLASSIF}"\n'
                    'if "haute_resolution" not in open(_gv, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ gemini_validator.py du 08/11 (2) absent du dépôt GitHub")\n'
                    '_pp = next(iter(_g_res.glob("/kaggle/working/ProjectAIMika/**/pipeline.py", recursive=True)), None)\n'
                    'if not _pp or "[ESCALADE]" not in open(_pp, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ pipeline.py du 08/11 (escalade) absent du dépôt GitHub")\n'
                    'if not _g_res.glob("/kaggle/working/ProjectAIMika/**/attribution_equipes.py", recursive=True) or '
                    '"Step 11b" not in open(_pp, encoding="utf-8").read():\n'
                    '    raise SystemExit("❌ ai/attribution_equipes.py et pipeline.py du 09/11 absents du dépôt GitHub")', 1)
assert "SCOUTIA_MEDIA_RESOLUTION_CLASSIF" in CODE, "point d'insertion introuvable"
CODE = f'import os as _os_tier\n_os_tier.environ["SCOUTIA_GEMINI_TIER"] = "{TIER}"\n' + CODE
CODE = CODE.replace("/ 1e6 * 0.75", "/ 1e6 * 0.375").replace("/ 1e6 * 3.75", "/ 1e6 * 1.875")   # tarif Flex
_CONTROLE_TIER = '''
_src_gv_t = open(_gv, encoding="utf-8").read()
if "SCOUTIA_GEMINI_TIER" not in _src_gv_t:
    raise SystemExit("❌ gemini_validator.py du 12/11 (mode Flex) absent du dépôt GitHub — rien n'est payé")
'''
CODE = CODE.replace('os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION', _CONTROLE_TIER + 'os.environ["SCOUTIA_MEDIA_RESOLUTION"] = RESOLUTION', 1)


import subprocess
# Chaque match tourne dans un PROCESSUS PYTHON SÉPARÉ : le cache Gemini, ses fichiers de sortie et les modules du
# pipeline repartent de zéro à chaque match (sinon l'état du 1er match se mélangerait aux suivants).
BILAN = {}
for m in MATCHS_A_TESTER:
    t0 = time.time()
    print(f"\n{'#' * 90}\n# MATCH {m} — résolution {RESOLUTION}\n{'#' * 90}", flush=True)
    code_m = re.sub(r'^MATCH = .*$', f'MATCH = "{m}"', CODE, count=1, flags=re.M)
    f = f"/kaggle/working/_test_resolution_{m}.py"
    open(f, "w", encoding="utf-8").write(code_m)
    proc = subprocess.Popen([sys.executable, "-u", f], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/kaggle/working")
    lignes = []
    for l in proc.stdout:
        sys.stdout.write(l); lignes.append(l)
    proc.wait()
    sortie = "".join(lignes)
    _t = re.search(r"Niveaux de service des questions payées : (\{[^}]*\})", sortie)
    print(f"   ⏱ {m} : durée totale {(time.time() - t0) / 60:.0f} min | niveaux de service : {_t.group(1) if _t else 'non affichés'}")
    b = re.search(r"BILAN DÉTECTION : (\d+)/(\d+) vrais buts \| (\d+) faux", sortie)
    c = re.search(r"TOTAL ≈ ([\d.]+) \$", sortie)
    manques = re.findall(r"but réel (\S+) : ❌ MANQUÉ", sortie)
    esc = re.search(r"\[ESCALADE\] (\d+) cas", sortie)
    att = re.search(r"ATTRIBUTION PIPELINE : tenues (\S+).*?justes (\d+) \| FAUX (\d+) \| non attribués (\d+) \| coût ([\d.]+) \$", sortie)
    changes = re.findall(r"\[ESCALADE\] (\S+) \(classification [^)]*\) : résolution réduite (\S+) → normale (\S+)", sortie)
    BILAN[m] = dict(trouves=int(b.group(1)) if b else None, vrais=int(b.group(2)) if b else None,
                    faux=int(b.group(3)) if b else None, cout=float(c.group(1)) if c else None,
                    manques=manques, minutes=(time.time() - t0) / 60, esc=int(esc.group(1)) if esc else 0,
                    changes=[f"{t} {a}→{b}" for t, a, b in changes if a != b],
                    att=(att.group(1), int(att.group(2)), int(att.group(3)), int(att.group(4)), float(att.group(5))) if att else None)

print(f"\n{'═' * 90}\nBILAN — résolution {RESOLUTION} sur {len(BILAN)} match(s)\n{'═' * 90}")
print("   match           buts trouvés   faux   référence (trouvés, faux)   coût     durée   escalades  manqués")
tot_t = tot_v = tot_c = 0; perdus = []; trop_faux = []
for m, r in BILAN.items():
    ref = REFERENCE.get(m, (None, None))
    if r["trouves"] is None:
        print(f"   {m:15s} — résultat non lu (voir la sortie du match)"); continue
    tot_t += r["trouves"]; tot_v += r["vrais"]; tot_c += r["cout"] or 0
    if ref[0] is not None and r["trouves"] < ref[0]:
        perdus.append(m)
    if ref[0] is None and r["trouves"] < r["vrais"]:
        perdus.append(m + " (à vérifier : pas de référence récente)")
    if ref[1] is not None and r["faux"] > ref[1] + 1:
        trop_faux.append(m)
    print(f"   {m:15s} {r['trouves']:>3d} / {r['vrais']:<3d}       {r['faux']:>3d}    {str(ref):26s}  "
          f"{(r['cout'] or 0):5.2f} $  {r['minutes']:4.0f} min  {r['esc']:5d}      {', '.join(r['manques'])}"
          + (f"  | changés : {', '.join(r['changes'])}" if r['changes'] else ""))
n_esc = sum(r["esc"] for r in BILAN.values())
_at = [r["att"] for r in BILAN.values() if r.get("att")]
print(f"\n   ATTRIBUTION (faite par le pipeline) : tenues OK {sum(1 for a in _at if a[0] == 'OK')}/{len(_at)} | "
      f"justes {sum(a[1] for a in _at)} | FAUX {sum(a[2] for a in _at)} | non attribués {sum(a[3] for a in _at)} | "
      f"coût {sum(a[4] for a in _at):.2f} $  (référence : normale, 41 justes / 0 faux / 2 non attribués sur 43 vrais buts)")
print("   CRITÈRES ATTRIBUTION (fixés le 11/11) : FAUX = 0 | non attribués ≤ 4 (sur les vrais buts trouvés) | "
      "aucune ligne « ÉCHECS » > 0 dans le détail par match (sinon run non valable pour l'attribution)")
print(f"\n   TOTAL : {tot_t} / {tot_v} vrais buts | coût Gemini mesuré ≈ {tot_c:.2f} $ (sans l'attribution des équipes)")
print(f"   ⚠️ les {n_esc} escalades en résolution normale ont pu être relues GRATUITEMENT dans l'ancien cache : en production,"
      f" compter ≈ {n_esc * 0.017:.2f} $ de plus au total (≈ 0,017 $ par question KO en résolution normale)")
print(f"   matchs avec un vrai but de moins que la référence : {perdus or 'aucun'}")
print(f"   matchs avec plus d'1 faux but de plus que la référence : {trop_faux or 'aucun'}")
ok = not [p for p in perdus if 'à vérifier' not in p] and not trop_faux
print(f"   → {'RÉUSSI : la résolution « medium » peut devenir le réglage par défaut' if ok else 'ÉCHEC ou à examiner : voir les matchs listés'}")


##########################################################################################
# MATCH minerois_ster — résolution medium
##########################################################################################
  Clés lues dans /kaggle/input/datasets/michaelziant/mziparameter/param.env : SECRET_KEY, CLAUDE_API_KEY, GEMINI_API_KEY, STRIPE_SECRET_KEY, STRIPE_PUBLIC_KEY, R2_SECRET_ACCESS_KEY, RUNPOD_API_KEY

ÉTAPE 1/3 — Clone
Cloning into '/kaggle/working/ProjectAIMika'...
Clone OK

ÉTAPE 2/3 — Dépendances
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.1 MB/s eta 0:00:00
   ━